# 08 | matplotlib 金融可视化 —— K线图、收益率分布、多子图与中文

## 这一讲做什么？

你已经有数据了（05、07），也学会了 Pandas 操作（04、06），接下来要**把数据变成图**。

金融领域的图表有自己的一套标准：K线图、收益率分布、多面板仪表盘。这一讲从零开始教你画。

### matplotlib 是什么？

matplotlib 是 Python 最基础的画图库。它的设计理念是：**给你最大的控制权**。
你可以控制图上每一个像素——代价是代码比 seaborn/plotly 长。
但学会它之后，其他画图库都是它的「快捷方式」。

**学习目标**
- 解决 matplotlib 中文乱码，掌握 macOS/Windows 字体配置
- 用纯 matplotlib 手绘 K 线图，理解 OHLC 数据结构
- 掌握 mplfinance 快速出图，理解它与手绘方案的适用边界
- 从收益率直方图到 Q-Q 图，建立「分布诊断」的完整视角
- 掌握 subplots / GridSpec 布局，搭建专业的多面板仪表盘

---


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import matplotlib.font_manager as fm
from matplotlib.gridspec import GridSpec

import mplfinance as mpf
import yfinance as yf
from scipy import stats
import seaborn as sns

np.random.seed(42)

print(f"mplfinance  {mpf.__version__}")
print(f"可用样式: {mpf.available_styles()}")
print("✓ 所有库导入成功")

---

## 3.1 中文显示配置 —— 先解决「口口口」

matplotlib 默认用 DejaVu Sans 渲染文字，它没有 CJK 字形，中文全部变成方块。

### 3.1.1 诊断当前环境字体

In [ ]:
import re
chinese_kw = ['PingFang', 'Heiti', 'Songti', 'Kaiti', 'Libian', 'Baoli',
              'Noto.*CJK', 'WenQuanYi', 'SimHei', 'SimSun']

all_fonts = fm.fontManager.ttflist
print(f"matplotlib 已索引 {len(all_fonts)} 个字体\n")

found = []
for f in all_fonts:
    for kw in chinese_kw:
        if re.search(kw, f.name, re.IGNORECASE):
            found.append(f)
            break

if found:
    print("可用的中文字体：")
    for f in found:
        print(f"  {f.name:30s} → {f.fname}")
else:
    print("⚠️ 未找到中文字体")

### 3.1.2 全局配置

```python
plt.rcParams['font.sans-serif'] = ['PingFang SC', 'Heiti SC', ...]
plt.rcParams['axes.unicode_minus'] = False  # 负号用 ASCII '-' 代替 Unicode '−'
```

In [ ]:
plt.rcParams['font.sans-serif'] = ['PingFang SC', 'Heiti SC', 'STHeiti', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False

fig, ax = plt.subplots(figsize=(6, 1.2))
ax.text(0.5, 0.5, '中文测试 · 收益率% · −0.05 · 沪深300',
        transform=ax.transAxes, ha='center', va='center', fontsize=14)
ax.set_title('matplotlib 中文显示验证')
ax.axis('off')
plt.show()

### 3.1.3 局部字体设置

In [ ]:
cn_font = fm.FontProperties(fname='/System/Library/Fonts/STHeiti Medium.ttc', size=14)

fig, ax = plt.subplots()
ax.set_title('局部字体设置', fontproperties=cn_font)
ax.set_xlabel('交易日', fontproperties=cn_font)
ax.set_ylabel('价格（元）', fontproperties=cn_font)
ax.plot([1, 2, 3], [10, 12, 9], 'o-')
plt.show()

---

## 3.2 K 线图 —— 从手绘到 mplfinance

两种方式：① 手绘理解底层映射 ② mplfinance 专业出图

两者都掌握后，日常浏览用 mplfinance，需要标注事件/箭头时用手绘。

### 3.2.1 数据准备

In [ ]:
ticker = '600519.SS'
df = yf.download(ticker, period='3mo', interval='1d', progress=False)

if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

print(f"维度: {df.shape}, 日期: {df.index[0].date()} → {df.index[-1].date()}")
df.head()

### 3.2.2 手绘 K 线 —— 理解每一根蜡烛

```
         ┬  High
    ┌────┴────┐
    │  实体    │  Cls>Opn 红涨, 反之为绿跌
    └────┬────┘
         ┴  Low
```

In [ ]:
def draw_candlestick_manual(ax, df, width=0.6, colorup='#DC143C', colordown='#228B22'):
    """Rectangle + Line2D 逐根手绘 K 线。"""
    dates = mdates.date2num(df.index.to_pydatetime())
    for date, row in zip(dates, df.itertuples()):
        if row.Close >= row.Open:
            color, bot, h = colorup, row.Open, row.Close - row.Open
        else:
            color, bot, h = colordown, row.Close, row.Open - row.Close
        ax.plot([date, date], [row.Low, row.High], color=color, lw=0.8)
        ax.add_patch(mpatches.Rectangle(
            (date - width/2, bot), width, max(h, 0.01),
            facecolor=color if h > 0 else 'none', edgecolor=color, lw=0.8))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

fig, ax = plt.subplots(figsize=(14, 6))
draw_candlestick_manual(ax, df.iloc[-30:])
ax.set_title(f'{ticker} 贵州茅台 — 最近 30 日（手绘）', fontsize=14)
ax.set_ylabel('价格（元）'); ax.grid(True, alpha=0.3)
fig.tight_layout(); plt.show()

### 3.2.3 mplfinance 基础出图

```python
mpf.plot(df, type='candle', style='charles', mav=(5, 20), volume=True)
```

In [ ]:
# 基础 K 线 + 成交量
mpf.plot(df.iloc[-60:], type='candle', style='charles',
         title=f'{ticker} 贵州茅台 — 最近 60 日',
         ylabel='价格（元）', volume=True,
         figsize=(16, 7), warn_too_much_data=200)

### 3.2.4 mplfinance 叠加 MA + MACD

In [ ]:
# MACD 指标
ema12 = df['Close'].ewm(span=12).mean()
ema26 = df['Close'].ewm(span=26).mean()
macd_line = ema12 - ema26
signal_line = macd_line.ewm(span=9).mean()
macd_hist = macd_line - signal_line

apds = [
    mpf.make_addplot(macd_line[-60:], panel=2, color='blue', width=0.8, ylabel='MACD'),
    mpf.make_addplot(signal_line[-60:], panel=2, color='orange', width=0.8),
    mpf.make_addplot(macd_hist[-60:], type='bar', panel=2, color='dimgray', alpha=0.5),
]

mpf.plot(df.iloc[-60:], type='candle', style='charles', mav=(5, 20, 60),
         volume=True, addplot=apds,
         title=f'{ticker} 贵州茅台 — K线+MA+MACD',
         ylabel='价格（元）', figsize=(16, 9),
         panel_ratios=(3, 1, 2), warn_too_much_data=200)

### 3.2.5 样式切换对比

In [ ]:
styles = ['charles', 'tradingview', 'binance']
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, s in zip(axes, styles):
    mpf.plot(df.iloc[-30:], type='candle', style=s,
             ax=ax, volume=False, ylabel='', warn_too_much_data=200)
    ax.set_title(f'style="{s}"', fontsize=12)
fig.suptitle('mplfinance 样式对比', fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout(); plt.show()

---

## 3.3 收益率分布

诊断工具链：直方图 + KDE → 正态叠加 → Q-Q 图

对数收益率 $r_t = \ln(P_t / P_{t-1})$，可加性好且更接近正态。

In [ ]:
returns = np.log(df['Close'] / df['Close'].shift(1)).dropna()

print(f"交易日: {len(returns)}")
print(f"均值: {returns.mean()*100:.4f}%  σ: {returns.std()*100:.4f}%")
print(f"偏度: {returns.skew():+.4f}  峰度: {returns.kurtosis():+.4f}")

jb_stat, jb_p = stats.jarque_bera(returns)
print(f"Jarque-Bera: {jb_stat:.2f}  p={jb_p:.6f} → {'拒绝正态 ✓' if jb_p < 0.05 else '无法拒绝'}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# 左：hist + KDE + 正态
ax1.hist(returns, bins=40, density=True, alpha=0.5, color='steelblue', edgecolor='white')
sns.kdeplot(returns, ax=ax1, color='darkblue', lw=2.5, label='KDE')
x = np.linspace(returns.min(), returns.max(), 500)
mu, sigma = returns.mean(), returns.std()
ax1.plot(x, stats.norm.pdf(x, mu, sigma), 'r--', lw=2,
         label=f'正态 N({mu*100:.2f}%, {sigma*100:.2f}%)')
ax1.set_title('收益率分布', fontsize=13)
ax1.set_xlabel('对数日收益率'); ax1.set_ylabel('密度')
ax1.legend(fontsize=9); ax1.grid(True, alpha=0.2)

# 右：Q-Q 图
stats.probplot(returns, dist='norm', plot=ax2)
ax2.set_title('Q-Q 图', fontsize=13)
ax2.set_xlabel('理论分位数'); ax2.set_ylabel('样本分位数')
ax2.grid(True, alpha=0.2)

fig.suptitle(f'{ticker} 贵州茅台 — 收益率分布诊断', fontsize=15, fontweight='bold', y=1.02)
fig.tight_layout(); plt.show()

In [ ]:
# 三只股票 KDE 对比
tickers = {'600519.SS': '贵州茅台', '600036.SS': '招商银行', '300750.SZ': '宁德时代'}
returns_dict = {}
for code, name in tickers.items():
    data = yf.download(code, period='6mo', interval='1d', progress=False)
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.get_level_values(0)
    r = np.log(data['Close'] / data['Close'].shift(1)).dropna()
    returns_dict[name] = r
    print(f"{name:6s}  μ={r.mean()*100:+.3f}%  σ={r.std()*100:.3f}%  "
          f"S={r.skew():+.3f}  K={r.kurtosis():+.3f}")

fig, ax = plt.subplots(figsize=(14, 6))
for (name, r), c in zip(returns_dict.items(), ['#DC143C', '#228B22', '#FF8C00']):
    sns.kdeplot(r, ax=ax, color=c, lw=2.5, label=f'{name} (σ={r.std()*100:.2f}%)')
ax.set_title('三只 A 股收益率 KDE 对比', fontsize=14, fontweight='bold')
ax.set_xlabel('对数日收益率'); ax.legend(fontsize=11)
ax.grid(True, alpha=0.2); fig.tight_layout(); plt.show()

---

## 3.4 多子图布局

| 层级 | API | 场景 |
|------|-----|------|
| 等分 | `plt.subplots()` | 2×2 面板 |
| 异形 | `GridSpec` | K线+成交量 |
| 内嵌 | `inset_axes` | 局部放大 |

2×2 四维面板① 收盘价② 日收益率③ 波动率④ 成交量

In [ ]:
df_6m = df.copy()
df_6m['Return'] = np.log(df_6m['Close'] / df_6m['Close'].shift(1))
df_6m['Vol20'] = df_6m['Return'].rolling(20).std() * np.sqrt(252)
df_6m = df_6m.dropna()
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
ax = axes[0, 0]
ax.plot(df_6m.index, df_6m['Close'], color='#DC143C', lw=1.2)
ax.fill_between(df_6m.index, df_6m['Close'].min(), df_6m['Close'], alpha=0.1, color='#DC143C')
ax.set_title('① 收盘价', fontsize=13, fontweight='bold')
ax.set_ylabel('价格（元）'); ax.grid(True, alpha=0.2)
ax = axes[0, 1]
colors = ['#DC143C' if r >= 0 else '#228B22' for r in df_6m['Return']]
ax.bar(range(len(df_6m)), df_6m['Return'], color=colors, width=0.8, alpha=0.7)
ax.axhline(y=0, color='black', lw=0.5)
ax.set_title('② 日收益率', fontsize=13, fontweight='bold')
ax.set_ylabel('收益率'); ax.grid(True, alpha=0.2); ax.set_xticks([])
ax = axes[1, 0]
ax.plot(df_6m.index, df_6m['Vol20'], color='steelblue', lw=1.5)
ax.fill_between(df_6m.index, 0, df_6m['Vol20'], alpha=0.15, color='steelblue')
ax.set_title('③ 20日滚动年化波动率', fontsize=13, fontweight='bold')
ax.set_ylabel('年化波动率'); ax.grid(True, alpha=0.2)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
ax = axes[1, 1]
vol_c = ['#DC143C' if row.Close >= row.Open else '#228B22' for row in df_6m.itertuples()]
ax.bar(df_6m.index, df_6m['Volume'], color=vol_c, width=0.8, alpha=0.6)
ax.set_title('④ 成交量', fontsize=13, fontweight='bold')
ax.set_ylabel('成交量'); ax.grid(True, alpha=0.2)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
fig.suptitle(f'{ticker} — 四维分析面板', fontsize=16, fontweight='bold', y=1.01)
fig.tight_layout(); plt.show()

In [ ]:
# GridSpec 不等分布局
fig = plt.figure(figsize=(18, 9))
gs = GridSpec(4, 6, figure=fig, hspace=0.35, wspace=0.4)

ax1 = fig.add_subplot(gs[0:3, 0:4])
ax1.plot(df_6m.index, df_6m['Close'], color='#DC143C', lw=1.2)
ax1.set_title('收盘价（宽面板）', fontsize=13, fontweight='bold')
ax1.set_ylabel('价格（元）'); ax1.grid(True, alpha=0.2)

ax2 = fig.add_subplot(gs[3, 0:4], sharex=ax1)
colors_bar = ['#DC143C' if r >= 0 else '#228B22' for r in df_6m['Return']]
ax2.bar(df_6m.index, df_6m['Return'], color=colors_bar, width=0.8, alpha=0.7)
ax2.axhline(y=0, color='black', lw=0.5)
ax2.set_title('日收益率', fontsize=13, fontweight='bold'); ax2.grid(True, alpha=0.2)

ax3 = fig.add_subplot(gs[0:2, 4:6])
ax3.hist(df_6m['Return'], bins=35, density=True, alpha=0.5, color='steelblue', edgecolor='white')
sns.kdeplot(df_6m['Return'], ax=ax3, color='darkblue', lw=2)
x_r = np.linspace(df_6m['Return'].min(), df_6m['Return'].max(), 200)
ax3.plot(x_r, stats.norm.pdf(x_r, df_6m['Return'].mean(), df_6m['Return'].std()),
         'r--', lw=1.5, alpha=0.7)
ax3.set_title('收益分布', fontsize=13, fontweight='bold'); ax3.grid(True, alpha=0.2)

ax4 = fig.add_subplot(gs[2:4, 4:6]); ax4.axis('off')
stats_data = [
    ['均值', f"{df_6m['Return'].mean()*100:.3f}%"],
    ['标准差', f"{df_6m['Return'].std()*100:.3f}%"],
    ['偏度', f"{df_6m['Return'].skew():.4f}"],
    ['峰度', f"{df_6m['Return'].kurtosis():.4f}"],
    ['夏普', f"{df_6m['Return'].mean()/df_6m['Return'].std()*np.sqrt(252):.3f}"],
    ['最大回撤', f"{(df_6m['Close']/df_6m['Close'].cummax()-1).min()*100:.2f}%"],
    ['胜率', f"{(df_6m['Return']>0).mean()*100:.1f}%"],
]
table = ax4.table(cellText=stats_data, colLabels=['指标', '数值'], loc='center',
                  cellLoc='center', colWidths=[0.35, 0.35])
table.auto_set_font_size(False); table.set_fontsize(11); table.scale(1, 1.5)
ax4.set_title('统计摘要', fontsize=13, fontweight='bold', y=1.05)

fig.suptitle('GridSpec 不等分布局', fontsize=16, fontweight='bold', y=1.01)
plt.show()

---

## 3.5 综合实战：六维仪表盘

K线+均线 | 收益分布+正态 | Q-Q图 | 成交量 | 波动率+累计收益（双Y轴）

① K线 + MA② 收益分布③ Q-Q④ 成交量⑤ 波动率 + 累计收益

In [ ]:
df_dash = df.iloc[-90:].copy()
df_dash['Return'] = np.log(df_dash['Close'] / df_dash['Close'].shift(1))
df_dash['CumReturn'] = df_dash['Return'].cumsum()
df_dash['Vol20'] = df_dash['Return'].rolling(20).std() * np.sqrt(252)
df_dash = df_dash.dropna()
dates = mdates.date2num(df_dash.index.to_pydatetime())
r_clean = df_dash['Return'].dropna()
fig = plt.figure(figsize=(20, 14))
gs = GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35, height_ratios=[2, 2, 1.5])
ax1 = fig.add_subplot(gs[0:2, 0:2])
for date, row in zip(dates, df_dash.itertuples()):
    if row.Close >= row.Open:
        color, bot, h = '#DC143C', row.Open, row.Close - row.Open
    else:
        color, bot, h = '#228B22', row.Close, row.Open - row.Close
    ax1.plot([date, date], [row.Low, row.High], color=color, lw=0.7)
    ax1.add_patch(mpatches.Rectangle((date - 0.3, bot), 0.6, max(h, 0.01),
                                      facecolor=color if h > 0 else 'none',
                                      edgecolor=color, lw=0.7))
ax1.plot(dates, df_dash['Close'].rolling(5).mean(), 'orange', lw=1, label='MA5')
ax1.plot(dates, df_dash['Close'].rolling(20).mean(), 'purple', lw=1, label='MA20')
ax1.set_title('① K线图 + 均线', fontsize=13, fontweight='bold')
ax1.legend(loc='upper left', fontsize=8); ax1.grid(True, alpha=0.2)
ax1.set_ylabel('价格'); ax1.tick_params(labelbottom=False)
ax2 = fig.add_subplot(gs[0, 2])
ax2.hist(r_clean, bins=30, density=True, alpha=0.45, color='steelblue', edgecolor='white')
sns.kdeplot(r_clean, ax=ax2, color='darkblue', lw=2)
x_l = np.linspace(r_clean.min(), r_clean.max(), 200)
ax2.plot(x_l, stats.norm.pdf(x_l, r_clean.mean(), r_clean.std()), 'r--', lw=1.5, alpha=0.7)
ax2.set_title('② 收益分布', fontsize=13, fontweight='bold'); ax2.grid(True, alpha=0.2)
ax3 = fig.add_subplot(gs[1, 2])
stats.probplot(r_clean, dist='norm', plot=ax3)
ax3.set_title('③ Q-Q 图', fontsize=13, fontweight='bold'); ax3.grid(True, alpha=0.2)
ax3.get_lines()[0].set_markersize(4); ax3.get_lines()[1].set_color('red')
ax4 = fig.add_subplot(gs[2, 0:2], sharex=ax1)
vol_c = ['#DC143C' if r >= 0 else '#228B22' for r in df_dash['Return']]
ax4.bar(dates, df_dash['Volume'].values, width=0.6, color=vol_c, alpha=0.6)
ax4.set_title('④ 成交量', fontsize=13, fontweight='bold')
ax4.set_ylabel('成交量'); ax4.grid(True, alpha=0.2)
ax4.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))
ax4.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
plt.setp(ax4.xaxis.get_majorticklabels(), rotation=45, ha='right')
ax5 = fig.add_subplot(gs[2, 2])
ax5.plot(df_dash.index, df_dash['Vol20'], color='steelblue', lw=1.5)
ax5.set_ylabel('波动率', color='steelblue', fontsize=10)
ax5.tick_params(axis='y', labelcolor='steelblue')
ax5b = ax5.twinx()
ax5b.plot(df_dash.index, df_dash['CumReturn'], color='#DC143C', lw=1.5, alpha=0.8)
ax5b.set_ylabel('累计收益', color='#DC143C', fontsize=10)
ax5b.tick_params(axis='y', labelcolor='#DC143C')
ax5b.axhline(y=0, color='gray', lw=0.5, ls='--')
ax5.set_title('⑤ 波动率 + 累计收益', fontsize=13, fontweight='bold')
ax5.grid(True, alpha=0.2)
ax5.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
plt.setp(ax5.xaxis.get_majorticklabels(), rotation=45, ha='right')
fig.suptitle(f'{ticker} 贵州茅台 — 六维仪表盘', fontsize=17, fontweight='bold', y=1.01)
plt.show()
print("✓ 仪表盘完成")

---

## 本讲小结

| 章节 | 学到什么 | 核心 API |
|------|---------|---------|
| 3.1 中文配置 | 字体回退链、全局/局部设置、负号修复 | `rcParams`, `FontProperties` |
| 3.2 K线图 | 手绘 OHLC→蜡烛 + mplfinance + 样式切换 + MACD | `Rectangle`, `mpf.plot()`, `make_addplot` |
| 3.3 收益率分布 | hist+KDE+正态叠加+Q-Q 诊断体系 | `sns.kdeplot`, `stats.probplot` |
| 3.4 多子图 | subplots/GridSpec/inset_axes | `GridSpec`, `twinx`, `sharex` |
| 3.5 实战 | 六面板仪表盘 | 以上全部 |

---

*2026-06-02*